# 01 — Data Exploration & Cleaning

Exploration and cleaning of the unified Cameroon agricultural dataset.

**Steps:**
1. Load & inspect raw data
2. Type casting & date parsing
3. Missing value analysis
4. Outlier detection & physical bounds validation
5. Duplicate removal
6. Cross-variable consistency checks
7. Export cleaned dataset

In [ ]:
import pandas as pd
import numpy as np
import sys, os

sys.path.insert(0, os.path.abspath('..'))
from utils.constants import CAMEROON_BOUNDS, AGROECOLOGICAL_ZONES

RAW_PATH = '../data/generated/cameroon_agricultural_unified.csv'
CLEAN_PATH = '../data/generated/cameroon_agricultural_cleaned.csv'

df = pd.read_csv(RAW_PATH, parse_dates=['observation_date'])
print(f'Loaded {len(df):,} rows x {df.shape[1]} cols')
df.head(3)

## 1 — Schema & dtypes

In [ ]:
df.info(show_counts=True)

In [ ]:
CAT_COLS = ['data_source', 'agroecological_zone', 'season', 'crop_name', 'crop_group']
for c in CAT_COLS:
    df[c] = df[c].astype('category')

df['irrigation_applied'] = df['irrigation_applied'].astype(bool)
df['data_quality_flag'] = df['data_quality_flag'].astype('int8')

print('Category counts:')
for c in CAT_COLS:
    print(f'  {c}: {df[c].nunique()} unique')

## 2 — Missing values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('pct', ascending=False)
if len(missing_df) == 0:
    print('No missing values found.')
else:
    print(missing_df)

## 3 — Duplicate check

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset=['record_id'])
n_dup = n_before - len(df)
print(f'Removed {n_dup} duplicate record_ids ({n_dup/n_before*100:.2f}%)')

## 4 — Physical bounds validation

Validate values against physical/agronomic limits for Cameroon.

In [ ]:
BOUNDS = {
    'latitude':            (CAMEROON_BOUNDS['south'], CAMEROON_BOUNDS['north']),
    'longitude':           (CAMEROON_BOUNDS['west'],  CAMEROON_BOUNDS['east']),
    'elevation':           (0, 4095),
    'temperature_min':     (-5, 45),
    'temperature_max':     (5, 50),
    'temperature_mean':    (0, 40),
    'precipitation_daily': (0, 500),
    'relative_humidity':   (0, 100),
    'solar_radiation':     (0, 35),
    'ph_water':            (3.5, 9.5),
    'organic_carbon':      (0.1, 10.0),
    'total_nitrogen':      (0.01, 1.0),
    'available_phosphorus': (1, 150),
    'sand_percentage':     (0, 100),
    'clay_percentage':     (0, 100),
    'fertilizer_nitrogen': (0, 400),
    'fertilizer_phosphorus': (0, 200),
    'organic_fertilizer':  (0, 50),
    'disease_incidence':   (0, 100),
    'yield_kg_ha':         (0, 200000),
    'biomass_kg_ha':       (0, 500000),
    'harvest_index':       (0.05, 0.95),
}

oob_report = []
for col, (lo, hi) in BOUNDS.items():
    if col in df.columns:
        oob = ~df[col].between(lo, hi)
        n = oob.sum()
        if n > 0:
            oob_report.append((col, n, f'{n/len(df)*100:.2f}%'))

if oob_report:
    for col, n, pct in oob_report:
        print(f'  {col}: {n} OOB ({pct})')
else:
    print('All values within bounds.')

In [ ]:
# Clamp out-of-bounds values
for col, (lo, hi) in BOUNDS.items():
    if col in df.columns:
        df[col] = df[col].clip(lo, hi)

print('All numeric columns clamped to physical bounds.')

## 5 — Cross-variable consistency

In [ ]:
# 5a. temperature_max must exceed temperature_min
bad_temp = df['temperature_max'] <= df['temperature_min']
print(f'temp_max <= temp_min: {bad_temp.sum()} rows')
df.loc[bad_temp, 'temperature_max'] = df.loc[bad_temp, 'temperature_min'] + 5.0
df['temperature_mean'] = ((df['temperature_min'] + df['temperature_max']) / 2).round(1)

# 5b. yield must not exceed biomass
bad_yield = df['yield_kg_ha'] > df['biomass_kg_ha']
print(f'yield > biomass: {bad_yield.sum()} rows')
df.loc[bad_yield, 'biomass_kg_ha'] = (df.loc[bad_yield, 'yield_kg_ha'] / df.loc[bad_yield, 'harvest_index']).round(0)

# 5c. sand + clay must not exceed 100
bad_texture = (df['sand_percentage'] + df['clay_percentage']) > 100
print(f'sand + clay > 100: {bad_texture.sum()} rows')
df.loc[bad_texture, 'sand_percentage'] = (100 - df.loc[bad_texture, 'clay_percentage'] - 10).clip(0, 100)

# 5d. Validate season vs latitude regime
bimodal_seasons = {'grand_dry_season','first_rainy_season','petit_dry_season','second_rainy_season','transition_to_dry'}
monomodal_seasons = {'dry_season','early_rainy','rainy_season','late_rainy'}
is_north = df['latitude'] >= 6.0
bad_s = ((~is_north) & (df['season'].isin(monomodal_seasons))).sum()
bad_n = ((is_north) & (df['season'].isin(bimodal_seasons))).sum()
print(f'Season/latitude mismatch: south={bad_s}, north={bad_n}')

print('Consistency checks done.')

## 6 — Distributions & descriptive stats

In [ ]:
print('=== Zone distribution ===')
print(df['agroecological_zone'].value_counts())
print()
print('=== Season distribution ===')
print(df['season'].value_counts())
print()
print('=== Crop distribution (top 15) ===')
print(df['crop_name'].value_counts().head(15))

In [ ]:
print('=== Mean yield (kg/ha) by zone ===')
print(df.groupby('agroecological_zone')['yield_kg_ha'].mean().round(0).sort_values(ascending=False))
print()
print('=== Mean temperature by zone ===')
print(df.groupby('agroecological_zone')['temperature_mean'].mean().round(1).sort_values(ascending=False))
print()
print('=== Mean humidity by zone ===')
print(df.groupby('agroecological_zone')['relative_humidity'].mean().round(1).sort_values(ascending=False))

## 7 — Key correlations

In [ ]:
corr_pairs = [
    ('relative_humidity', 'disease_incidence', 'humidity <-> disease'),
    ('elevation', 'temperature_mean', 'elevation <-> temperature'),
    ('precipitation_daily', 'relative_humidity', 'precip <-> humidity'),
    ('fertilizer_nitrogen', 'yield_kg_ha', 'nitrogen <-> yield'),
    ('organic_carbon', 'total_nitrogen', 'C <-> N'),
]

print('Key correlations (Pearson r):')
for c1, c2, label in corr_pairs:
    r = df[c1].corr(df[c2])
    print(f'  {label:30s}  r = {r:+.3f}')

## 8 — Export cleaned dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
size_mb = os.path.getsize(CLEAN_PATH) / (1024*1024)
print(f'Saved cleaned dataset: {len(df):,} rows, {size_mb:.1f} MB')
print(f'Path: {CLEAN_PATH}')